In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import re

sys.path.append(os.path.abspath(".."))
from src.data_processing import BaseDataProcessor
import src.data_cleaning as dc

### Data Extraction

In [2]:
src_file = r'..\data\interim\ultra_marathon_races_cleaned.csv'

In [3]:
um_races = BaseDataProcessor(src_file)
um_races.data.sample(5)

2026-07-12 19:17:29 - INFO - Data loaded successfully from ..\data\interim\ultra_marathon_races_cleaned.csv


,year,event_date_str,event_name,event_distance,event_number_of_finishers,athlete_finish_time_str,athlete_club,athlete_country,birth_year,gender,age_category,athlete_average_speed,athlete_id
4329228,2006,15.10.2006,River Shimanto 100km (JPN),100km,1043,8:26:04 h,博多陸友会,JPN,1969.0,M,M35,11.856,776240
2694327,2020,27.12.2020,Xisaiguo Ultra Trail - 50km (CHN),50km,118,10:02:51 h,NaN,CHN,1974.0,M,M45,4.976,289809
1205442,2017,18.02.2017,Bay Ultra 50 km (RSA),50km,300,6:23:55 h,Realgij,RSA,1979.0,M,M35,7.814,56421
3220175,2022,02.04.2022,Trencacims Paüls - Ultramarató (ESP),53km,179,11:00:07 h,TR Cabrafeixet EL Perel,ESP,1967.0,M,M50,4.817,962780
6927216,1980,06.-07.06.1980,100 km Lauf Biel (SUI),100km,2866,10:11:00 h,*Biel,SUI,1943.0,M,M35,9820.0,1522393


In [4]:
athlete_data = um_races.data

### Data Manipulation

In [5]:
athlete_data[['event_start_date', 'event_end_date', 'event_duration']] = dc.extract_event_dates(athlete_data)

In [6]:
athlete_data[['event_name', 'event_host_country']] = dc.extract_event_name(athlete_data.event_name)

In [7]:
athlete_data[['event_type', 'distance_km', 'timed_event_duration_min']] = dc.parse_event_distance(athlete_data.event_distance)

In [8]:
athlete_data.loc[:,'athlete_club'] = (
    athlete_data["athlete_club"]
    .str.strip().str.replace(r'^\*\s*', '', regex=True).str.title()
)

In [9]:
# def extract_athlete_info
athlete_data[["athlete_gender", "athlete_age_group"]] = (
    athlete_data["age_category"]
    .str.extract(r"([A-Za-z]+)(\d+)")
)

athlete_data["athlete_gender"] = (
    athlete_data["athlete_gender"]
    .str.replace("U", "", regex=False)
    .str.replace("W", "F", regex=False)
)

athlete_data.athlete_age_group = athlete_data.athlete_age_group.astype('float')

In [10]:
athlete_data["birth_year"] = (
    athlete_data.groupby("athlete_id")["birth_year"]
    .transform("max")
)

In [11]:
athlete_data['athlete_age'] = athlete_data.year - athlete_data.birth_year

In [12]:
len(athlete_data[athlete_data.athlete_age.isin(range(13,101))])/len(athlete_data)

0.9193024663104657

In [13]:
athlete_data.sample(5)

,year,event_date_str,event_name,event_distance,event_number_of_finishers,athlete_finish_time_str,athlete_club,athlete_country,birth_year,gender,...,event_start_date,event_end_date,event_duration,event_host_country,event_type,distance_km,timed_event_duration_min,athlete_gender,athlete_age_group,athlete_age
4331623,2006,09.10.2006,Echigo Kubikino 100km Ultramarathon,100km,727,10:37:37 h,NaN,JPN,1967.0,M,...,2006-10-09,2006-10-09,One-day,JPN,Distance,100.00,NaN,M,35.0,39.0
5133086,2011,25.-26.06.2011,Sundown 100 km Ultramarathon,100km,514,17:43:10 h,NaN,SGP,1984.0,M,...,2011-06-25,2011-06-26,2 days,SGP,Distance,100.00,NaN,M,23.0,27.0
4405035,2007,28.04.2007,Vasque Free State Trail 100 km Run,100km,11,10:34:14 h,"Olathe, Ks",USA,1979.0,M,...,2007-04-28,2007-04-28,One-day,USA,Distance,100.00,NaN,M,23.0,28.0
7402870,1994,29.05.1994,Isle of Man TT course 40 miler,39.5mi,11,5:22:57 h,NaN,GBR,1981.0,M,...,1994-05-29,1994-05-29,One-day,GBR,Distance,63.57,NaN,M,23.0,13.0
28575,2018,10.03.2018,Six Foot Track,45km,869,6:19:32 h,NaN,AUS,NaN,M,...,2018-03-10,2018-03-10,One-day,AUS,Distance,45.00,NaN,NaN,NaN,NaN


In [14]:
athlete_data['birth_year_temp'] = np.where(
    ~athlete_data.athlete_age.isin(range(13,101)),
    athlete_data.year - athlete_data.athlete_age_group, np.nan
)
athlete_data['birth_year'] = np.where(
    athlete_data.birth_year_temp.notna(), athlete_data.birth_year_temp, athlete_data.birth_year
)
athlete_data['athlete_age'] = np.where(
    athlete_data.birth_year_temp.notna(), athlete_data.athlete_age_group, athlete_data.athlete_age
)

athlete_data.loc[~athlete_data.athlete_age.isin(range(13,101)),'athlete_age'] = np.nan

In [15]:
athlete_data[(athlete_data.athlete_age.isna())][['year', 'birth_year', 'age_category', 'athlete_age_group', 'athlete_age', 'birth_year_temp']].isna().sum()

year                      0
birth_year           534510
age_category         541240
athlete_age_group    541240
athlete_age          541240
birth_year_temp      541240
dtype: int64

In [16]:
athlete_data_filter = athlete_data[athlete_data.athlete_age.notna()]

In [17]:
cols = [
    'year', 'event_date_str', 
    'event_start_date', 'event_end_date', 'event_duration', 
    'event_name', 'event_host_country', 
    'event_distance', 'event_type', 'distance_km', 'timed_event_duration_min', 

    'athlete_id', 'birth_year', 'gender', 
    'athlete_gender', 'birth_year_temp',    # drop fields
     
    
    'event_number_of_finishers', 'athlete_finish_time_str', 'athlete_club',
    'athlete_country',  'athlete_age', 'athlete_age_group', 
    'age_category',
    'athlete_average_speed', 
]

In [18]:
athlete_data_filter[cols].sample(5)

,year,event_date_str,event_start_date,event_end_date,event_duration,event_name,event_host_country,event_distance,event_type,distance_km,...,athlete_gender,birth_year_temp,event_number_of_finishers,athlete_finish_time_str,athlete_club,athlete_country,athlete_age,athlete_age_group,age_category,athlete_average_speed
6623703,2015,20.-21.06.2015,2015-06-20,2015-06-21,2 days,Ulsan Taewha River 100 Km Ultramarathon,KOR,100km,Distance,100.0,...,M,NaN,267,12:14:40 h,Yoda,KOR,34.0,23.0,M23,8.167
4738927,2009,08.03.2009,2009-03-08,2009-03-08,One-day,"Strasimeno, Ultramaratona del Trasimeno",ITA,58.7km,Distance,58.7,...,M,NaN,196,5:37:01 h,Fano Corre Pu,ITA,46.0,45.0,M45,10.451
4097708,2003,19.04.2003,2003-04-19,2003-04-19,One-day,Two Oceans Marathon,RSA,56km,Distance,56.0,...,M,NaN,6251,6:49:20 h,Muirite Striders,RSA,34.0,23.0,M23,8.208
1867501,2019,06.01.2019,2019-01-06,2019-01-06,One-day,Hillingdon Cycle Circuit,GBR,6h,Timed,NaN,...,M,NaN,17,72.259 km,NaN,POL,56.0,55.0,M55,12.043
1786837,2017,04.11.2017,2017-11-04,2017-11-04,One-day,Great Walker Nanjing - 50 km,CHN,48.5km,Distance,48.5,...,M,NaN,916,12:16:43 h,NaN,CHN,26.0,23.0,M23,3.95
